# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
import csv

with DATA_PATH.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

row = rows[0] if rows else {}

print(f"Filas: {len(rows)}")
print(f"Columnas: {list(row.keys())}")
print("Primera fila:", row)


Filas: 5
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Primera fila: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo? -> sample_id
2. ¿Cuántas características numéricas hay? -> 11
3. ¿Qué columna extra añadirías para probar el contrato? -> extra_column para comprobar el contrato es capaz de rechazar campos desconocidos
4. ¿Qué valor de `ph` sería claramente inválido? -> 5.0

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [10]:
from pydantic import BaseModel, ConfigDict, Field

class WineQualityRequestPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

wine_features = {key: value for key, value in row.items() if key != "sample_id"}
request = WineQualityRequestPydantic.model_validate(wine_features)

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [4]:
from pydantic import ValidationError

In [ ]:
# TODO: completa los casos y captura ValidationError.
try:
    WineQualityRequestPydantic.model_validate(wine_features)
    print("Fila válida")
except ValidationError as error:
    print(error)

#Columna extra
extra_row = wine_features.copy()
extra_row["unexpected_field"] = "valor"

try:
    WineQualityRequestPydantic.model_validate(extra_row)
except ValidationError as error:
    print(error)

#ph fuera de rango
invalid_row = wine_features.copy()
invalid_row["ph"] = 5.0

try:
    WineQualityRequestPydantic.model_validate(invalid_row)
except ValidationError as error:
    print(error)

Fila válida
1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='valor', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=5.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [12]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.
valid_dataclass = WineQualityRequestDataclass(**request.model_dump())
print(valid_dataclass)
invalid_data = request.model_dump()
invalid_data["ph"] = 99.0

WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4)


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en entrada de datos porque valida campos desconocidos, tipos y rangos. Usaría `dataclass` en la representacion interna porque agrupa datos ya validados facilmente.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [13]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

# TODO: construye vector usando getattr(request, name).
vector = [getattr(request, name) for name in FEATURE_NAMES]
print(FEATURE_NAMES)
print(vector)

('fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol')
[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores? -> Si
- ¿El clasificador detectaría necesariamente el error? -> No tiene porque, puede aceptar el vector y predecir incorrectamente
- ¿Cómo evitarías que cada script definiera un orden distinto? -> Definiendo un FEATURE_NAMES, y haciendo que cada script lo importe

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado? -> sample_id
- ¿Qué categoría se ha predicho? -> quality_band
- ¿Con qué confianza? -> entre 0 y 1, con confidence
- ¿Qué versión de modelo y preprocesado se utilizaron? -> preprocessing_version + model_version

In [14]:
from typing import Literal

class WineQualityPredictionPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")

    sample_id: str = Field(min_length=1)
    quality_band: Literal["acceptable", "excellent", "needs_review"]
    confidence: float = Field(ge=0, le=1)
    model_version: str = Field(min_length=1)
    preprocessing_version: str = Field(min_length=1)

prediction = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="needs_review",
    confidence=0.86,
    model_version="wine-quality-rf-v1",
    preprocessing_version="wine-red-features-v1",
)

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.